<a href="https://colab.research.google.com/github/shuvad23/Foundation-Introduction-to-LangChain---Python/blob/main/module_1_1_1_prompting_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install python-dotenv langchain langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.5 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

GEMINI_API_KEY =  userdata.get('GEMINI_API_KEY')
print(GEMINI_API_KEY[:6], "******")

AIzaSy ******


In [ ]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

print("Available Gemini models that support generateContent:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Gemini models that support generateContent:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0.5,
    max_output_tokens = 1024,
    google_api_key = GEMINI_API_KEY,
    verbose = True,
    streaming = True,
)

In [ ]:
response = llm.invoke("what is the capital of india")
print(response.content)

The capital of India is **New Delhi**.


In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=llm)
question = HumanMessage(content="what is the capital of india")
response = agent.invoke(
    {"messages":[question]}
)
print(response['messages'][1].content)

The capital of India is **New Delhi**.


In [ ]:
system_propmt = "You are a science fiction writer, create a capital city at the users request."
scifi_agent = create_agent(
    model = llm,
    system_prompt = system_propmt
)
response = scifi_agent.invoke(
    {"messages":[question]}
)
print(response['messages'][1].content)

Ah, you're asking about India! In our current timeline, of course, that would be **New Delhi**.

But if you're asking *me*, a humble chronicler of futures


## Few-shot examples

In [ ]:
system_prompt1 = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model = llm,
    system_prompt = system_prompt1
)

response = scifi_agent.invoke(
    {"messages":[question]}
)
print(response['messages'][1].content)

Bharatanova


## Structured prompts

In [ ]:
system_prompt2 = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model = llm,
    system_prompt = system_prompt2
)
respone = scifi_agent.invoke(
    {"messages":[question]}
)
print(respone['messages'][1].content)


As a science fiction writer, my expertise is in crafting futuristic space cities, not in current Earth geography!

However, if you're looking for a **space capital city** that might one day serve as the heart of a future Indian interstellar empire, I can certainly create one for you:

---

**Name:** Bharatavarsha Prime

**Location:** Orbiting the exoplanet "Shanti" in the Kepler-186 system, a terraformed world rich in rare earth elements and crystalline structures. Bharatavarsha Prime is a massive, rotating toroidal habitat, with smaller orbital stations and asteroid mining outposts forming its extended metropolitan area.

**Vibe:** Serene, Technologically Advanced, Culturally Rich

**Economy:**
*   **Astro-Engineering & Terraforming:** Leading the galaxy in large-scale planetary engineering, atmospheric processors, and self-sustaining biodomes.
*   **Quantum Computing & AI Development:** Home to the most advanced quantum processors and ethical AI research hubs, driving interstellar co

## Structured output

In [ ]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

system_prompt3 = """First, extract all market size references from this research paper.
                Then, summarize them into a table: region | year | value | source. Next, project the
                2025 value using linear growth based on prior data. Do not proceed to projection until
                extraction is complete. Validate all numbers against listed sources. Present the output
                clearly for inclusion in a leadership deck for this client (Add name of client if the AI
                tool is internal and secure, and occasionally include mission, vision statements, and
                content from the website or the annual reports to follow a similar tone)."""
agent = create_agent(
    model = llm,
    system_prompt = system_prompt3,
    response_format=CapitalInfo
)
question = HumanMessage(content="what is the base foundation of german economy")
response = agent.invoke(
    {"messages":[question]}
)

response["structured_response"]

CapitalInfo(name='Germany', location='Central Europe', vibe='Innovative, precise, high-quality', economy='Highly developed social market economy, driven by exports, strong manufacturing sector (automotive, machinery, chemicals), skilled labor force, and robust R&D investment. Key sectors include automotive, mechanical engineering, chemicals, electrical engineering, and pharmaceuticals. It is characterized by a strong Mittelstand (small and medium-sized enterprises) and a focus on sustainability and digitalization.')

## A Mini Market analysis project


In [ ]:
# ============================================
# 1. Install Requirements
# ============================================
# pip install langchain  pydantic

from typing import List, Optional
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


# ============================================
# 2. Pydantic Schema
# ============================================

class MarketEntry(BaseModel):
    region: str = Field(..., description="Geographic region")
    year: int = Field(..., description="Year of market size")
    value: float = Field(..., description="Market value number")
    currency: str = Field(..., description="Currency like USD")
    source: str = Field(..., description="Citation or report name")


class MarketProjection(BaseModel):
    region: str
    projected_2025_value: float
    method: str = "Linear Growth"


class LeadershipDeckOutput(BaseModel):
    client_name: Optional[str]
    mission_statement: Optional[str]
    vision_statement: Optional[str]
    extracted_data: List[MarketEntry]
    projections: List[MarketProjection]
    validation_notes: Optional[str] = "No validation notes provided."


# ============================================
# 3. LLM Setup
# ============================================

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0.5,
    max_output_tokens = 1024,
    google_api_key = GEMINI_API_KEY,
    verbose = True,
    streaming = True,
)



system_prompt_new = """
You are a highly meticulous market intelligence analyst. Your primary goal is to extract, analyze, and present market data in an extremely precise and structured format, suitable for a leadership deck.

**Your final output MUST be a JSON object that strictly conforms to the `LeadershipDeckOutput` Pydantic schema, ensuring ALL fields (`client_name`, `mission_statement`, `vision_statement`, `extracted_data`, `projections`, `validation_notes`) are populated.**

Follow these steps rigorously:

1.  **Extract Data (for `extracted_data` field - List[MarketEntry])**:
    *   Thoroughly read the 'Research Paper'.
    *   Identify *every single* market size reference. For each, you *must* create a complete `MarketEntry` object.
    *   For *each* `MarketEntry`, ensure `region`, `year`, `value` (as float), `currency` (e.g., 'USD'), and `source` are all present.
    *   **Crucial for completeness**: If `currency` or `source` is not directly adjacent to a `value`, infer it from the closest explicit mention within the same sentence or paragraph, or from other entries for the same region/report if consistent. Do NOT leave any `MarketEntry` field empty or missing.

2.  **Calculate Projections (for `projections` field - List[MarketProjection])**:
    *   **This list is REQUIRED.** If a region has less than two data points in `extracted_data`, no projection is possible for that region, but the `projections` list should still be present (it may be empty).
    *   For *every* `region` found in `extracted_data` that has at least two data points, perform a linear growth projection to calculate its market value for the year 2025.
    *   Create a `MarketProjection` object for each valid projection, including `region`, `projected_2025_value` (as float), and `method` set to 'Linear Growth'.
    *   *Linear Growth Calculation*: If market value is `X` in `Year_A` and `Y` in `Year_B`, the annual growth rate = `(Y - X) / (Year_B - Year_A)`. The projected value for 2025 = `Y + (2025 - Year_B) * annual growth rate`. **Ensure this calculation is performed and `projected_2025_value` is always included.**

3.  **Provide Validation Notes (for `validation_notes` field - str)**:
    *   **This field is REQUIRED and MUST be a string.**
    *   Detail any specific assumptions made (e.g., inferring currency/source), discrepancies found in the source text, or challenges in extracting data.
    *   If all data was explicit and no assumptions were needed, state: 'All data validated against provided sources; no discrepancies or inferences required.' **Ensure this field is always populated with a string.**

4.  **Populate Client Info (`client_name`, `mission_statement`, `vision_statement` - Optional[str])**:
    *   Populate these accurately from the input context.

"""


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_new),
        ("human", "{input}")
    ]
)


# ============================================
# 4. Agent Function
# ============================================

def run_market_agent(paper_text: str, client_name="HelixMedica AI"):
    chain = prompt | llm.with_structured_output(LeadershipDeckOutput)

    response = chain.invoke({
        "input": f"""
        Research Paper:
        {paper_text}

        Client Name: {client_name}
        Mission: Improve global healthcare with AI.
        Vision: Accessible AI for every patient.
        """
    })

    return response


# ============================================
# 5. Example Research Paper
# ============================================

paper = """
The global AI healthcare market reached USD 12.5 billion in 2022
and is expected to grow from USD 15.3 billion in 2023
according to McKinsey 2023 report.

In Asia-Pacific, the market was valued at USD 3.1 billion in 2021
and USD 4.0 billion in 2022 (Statista Healthcare AI Report).
"""

result = run_market_agent(paper)

print(result.model_dump_json(indent=2))

Error processing data: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 31.859890287s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 

In [ ]:
# ==========================================================
# MARKET INTELLIGENCE AGENT
# Production Ready - Gemini + LangChain + Pydantic
# For HelixMedica AI Research Assistant
# ==========================================================

# pip install langchain langchain-google-genai pydantic python-dotenv

import os
from typing import List, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


# ==========================================================
# 1. Load API KEY
# ==========================================================
# load_dotenv()
# GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# if not GEMINI_API_KEY:
#     raise ValueError("❌ GEMINI_API_KEY not found in .env file")


# ==========================================================
# 2. Pydantic Schemas
# ==========================================================

class MarketEntry(BaseModel):
    region: str = Field(..., description="Geographic region")
    year: int = Field(..., description="Year of market size")
    value: float = Field(..., description="Market value")
    currency: str = Field(..., description="Currency like USD")
    source: str = Field(..., description="Citation or report name")


class MarketProjection(BaseModel):
    region: str
    projected_2025_value: float
    method: str = "Linear Growth"


class LeadershipDeckOutput(BaseModel):
    client_name: str
    mission_statement: str
    vision_statement: str
    extracted_data: List[MarketEntry]
    projections: List[MarketProjection]
    validation_notes: Optional[str] = "All data validated automatically."


# ==========================================================
# 3. LLM Setup
# ==========================================================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
    max_output_tokens=1024,
    google_api_key=GEMINI_API_KEY,
)


SYSTEM_PROMPT = """
You are a meticulous market intelligence analyst.

Steps:
1. Extract ALL market size references from research paper.
2. Validate numbers vs sources.
3. THEN project 2025 values using linear growth.
4. Format for leadership deck tone.

IMPORTANT:
Return JSON that EXACTLY matches LeadershipDeckOutput schema.
ALWAYS include validation_notes field.
"""


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "{input}")
    ]
)


# ==========================================================
# 4. Agent Function
# ==========================================================

def run_market_agent(
    paper_text: str,
    client_name: str = "HelixMedica AI",
    mission: str = "Improve global healthcare with AI.",
    vision: str = "Accessible AI for every patient."
) -> LeadershipDeckOutput:

    chain = prompt | llm.with_structured_output(LeadershipDeckOutput)

    try:
        result = chain.invoke({
            "input": f"""
Research Paper:
{paper_text}

Client Name: {client_name}
Mission: {mission}
Vision: {vision}
"""
        })

        return result

    except Exception as e:
        print("❌ Agent failed:", e)
        raise


# ==========================================================
# 5. Example Paper
# ==========================================================

if __name__ == "__main__":

    sample_paper = """
    The global AI healthcare market reached USD 12.5 billion in 2022
    and USD 15.3 billion in 2023 according to McKinsey 2023 report.

    Asia-Pacific market was USD 3.1 billion in 2021 and
    USD 4.0 billion in 2022 (Statista Healthcare AI Report).
    """

    output = run_market_agent(sample_paper)

    print("\n✅ STRUCTURED OUTPUT\n")
    print(output.model_dump_json(indent=2))


✅ STRUCTURED OUTPUT

{
  "client_name": "HelixMedica AI",
  "mission_statement": "Improve global healthcare with AI.",
  "vision_statement": "Accessible AI for every patient.",
  "extracted_data": [
    {
      "region": "Global",
      "year": 2022,
      "value": 12.5,
      "currency": "USD",
      "source": "McKinsey 2023 report"
    },
    {
      "region": "Global",
      "year": 2023,
      "value": 15.3,
      "currency": "USD",
      "source": "McKinsey 2023 report"
    },
    {
      "region": "Asia-Pacific",
      "year": 2021,
      "value": 3.1,
      "currency": "USD",
      "source": "Statista Healthcare AI Report"
    },
    {
      "region": "Asia-Pacific",
      "year": 2022,
      "value": 4.0,
      "currency": "USD",
      "source": "Statista Healthcare AI Report"
    }
  ],
  "projections": [
    {
      "region": "Global",
      "projected_2025_value": 18.4,
      "method": "Linear growth projection based on 2022-2023 data"
    },
    {
      "region": "Asia-Pac